# 05 — General HRV characterization

**Purpose:** describe the distribution, variability, asymmetry, and extreme values of each HRV measurement in the final cohort.

**Input:** `../../outputs/dataset_final.csv` (Notebook 03 output).

**Output:** `../../outputs/05_general_hrv_characterization/hrv_descriptive_statistics.csv`.

This notebook does **not** clean, transform, filter, or modify participant data. Outliers are flagged using the 1.5 × IQR and 3 × IQR rules for description only.

In [4]:
from pathlib import Path
import pandas as pd
from IPython.display import display

INPUT_FILE = Path("../../outputs/dataset_final.csv")
OUTPUT_DIR = Path("../../eda_results/05_general_hrv_characterization")

hrv_columns = [
    "t2m_pre_mean_rr",
    "t2m_pre_mean_hr",
    "t2m_pre_sdnn",
    "t2m_pre_rmssd",
    "t2m_pre_hf",
    "t2m_pre_lf",
    "t2m_pre_vlf",
]

df = pd.read_csv(INPUT_FILE)
missing_columns = [column for column in hrv_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing HRV columns: {missing_columns}")
if not all(pd.api.types.is_numeric_dtype(df[column]) for column in hrv_columns):
    raise TypeError("All HRV columns must contain numeric data.")

print(f"Participants: {len(df)}")
print(f"HRV variables: {len(hrv_columns)}")


Participants: 476
HRV variables: 7


## Descriptive statistics

One row per HRV variable. The CSV includes central tendency, dispersion, quantiles, skewness, excess kurtosis, and counts of values outside Tukey IQR fences. These fences are **descriptive flags**, not clinical thresholds or exclusion criteria.

In [5]:
results = []

for column in hrv_columns:
    values = df[column].dropna()
    q01, q05, q25, q75, q95, q99 = values.quantile([0.01, 0.05, 0.25, 0.75, 0.95, 0.99])
    iqr = q75 - q25
    mild_low, mild_high = q25 - 1.5 * iqr, q75 + 1.5 * iqr
    extreme_low, extreme_high = q25 - 3 * iqr, q75 + 3 * iqr

    results.append({
        "hrv_variable": column,
        "n_total": len(df),
        "n_valid": len(values),
        "n_missing": df[column].isna().sum(),
        "mean": values.mean(),
        "median": values.median(),
        "std": values.std(),
        "min": values.min(),
        "p01": q01,
        "p05": q05,
        "q25": q25,
        "q75": q75,
        "p95": q95,
        "p99": q99,
        "max": values.max(),
        "iqr": iqr,
        "skewness": values.skew(),
        "excess_kurtosis": values.kurt(),
        "n_outliers_1_5_iqr": ((values < mild_low) | (values > mild_high)).sum(),
        "n_extreme_3_iqr": ((values < extreme_low) | (values > extreme_high)).sum(),
    })

summary = pd.DataFrame(results)
display(summary.round(3))


,hrv_variable,n_total,n_valid,n_missing,mean,median,std,min,p01,p05,q25,q75,p95,p99,max,iqr,skewness,excess_kurtosis,n_outliers_1_5_iqr,n_extreme_3_iqr
0,t2m_pre_mean_rr,476,476,0,860.193,856.0,124.429,468.0,586.000,661.750,776.75,950.000,1057.250,1129.25,1296.0,173.250,0.023,0.017,2,0
1,t2m_pre_mean_hr,476,476,0,71.315,70.0,11.035,46.0,53.000,56.750,63.00,77.000,91.000,102.50,128.0,14.000,1.008,2.145,11,1
2,t2m_pre_sdnn,476,476,0,136.763,142.5,93.525,4.8,6.725,9.775,71.00,194.250,285.750,355.75,626.0,123.250,0.829,2.429,4,2
3,t2m_pre_rmssd,476,476,0,14.691,13.7,8.540,0.5,1.275,4.975,9.80,17.925,26.275,43.50,71.6,8.125,2.832,15.595,14,5
4,t2m_pre_hf,476,476,0,419.706,93.5,1225.133,1.0,3.000,9.000,34.75,259.250,1650.500,5772.50,13160.0,224.500,6.508,52.003,62,42
5,t2m_pre_lf,476,476,0,454.418,116.5,1193.647,7.0,10.750,15.750,56.00,259.500,2145.000,6915.00,10220.0,203.500,5.256,31.836,66,49
6,t2m_pre_vlf,476,476,0,118.849,30.5,276.082,1.0,2.000,4.000,15.00,82.000,564.500,1452.50,2340.0,67.000,4.573,24.747,66,44


In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_file = OUTPUT_DIR / "hrv_descriptive_statistics.csv"
summary.to_csv(output_file, index=False)
print(f"Saved: {output_file}")


Saved: ../../eda_results/05_general_hrv_characterization/hrv_descriptive_statistics.csv
